barrierQP solves a dense, strictly convex quadratic program with an
infeasible-start Mehrotra predictor-corrector. Each iteration factors one
reduced KKT matrix and asks that single factorization two questions: where
pure Newton would go (the affine predictor), and where the iteration should
actually go once the predictor has measured how much complementarity survives
(the centered corrector). This notebook runs the solver, then rebuilds every
claim against an independent full-system reference.

In [1]:
import fractions

import numpy as np
from numpy.testing import assert_allclose
import osqp
from scipy import sparse

import jax

jax.config.update("jax_enable_x64", True)  # before any JAX array exists: float32 would blur every gate below

import jax.numpy as jnp

import barrierqp

assert jnp.ones(1).dtype == jnp.float64
np.set_printoptions(precision=9, suppress=True)

The fixture is a pentagon: the square $-1 \le x_1, x_2 \le 1$ cut by
$0.625\,(x_1 + x_2) \le 1$. The quadratic objective is chosen so the optimum
$x^* = (0.85, 0.75)$ lies on the diagonal cut alone, with that constraint's
dual exactly one. Because every $h_i = 1$, the solver's fixed start
$x = 0$, $s = z = \mathbf{1}$ satisfies $s = h - Gx$ exactly, so the run
begins on the inequality manifold with unit complementarity.

In [2]:
# Pentagon g_i . x <= 1: the square [-1, 1]^2 cut by 0.625*(x1 + x2) <= 1.
G = np.array([[1.0, 0.0], [0.0, 1.0], [0.625, 0.625], [-1.0, 0.0], [0.0, -1.0]])
h = np.ones(5)
P = np.array([[2.0, 0.5], [0.5, 1.5]])
# q makes the KKT conditions exact at x* = (0.85, 0.75) with z* = (0, 0, 1, 0, 0):
# P x* + q + G.T z* = [2.075, 1.55] - [2.7, 2.175] + [0.625, 0.625] = 0.
q = np.array([-2.7, -2.175])
A, b = np.zeros((0, 2)), np.zeros(0)  # no equality rows in this fixture
x_star = np.array([0.85, 0.75])

solver = barrierqp.Solver(P, q, A, b, G, h)
result, trace = solver.trace()

assert result.status == "solved"
N = result.iterations
assert N == 6 and result.factorizations == N and result.newton_solves == 2 * N

# Iterates 0..N: each iteration's recorded start plus the accepted end.
x_path = np.vstack([np.asarray(trace.x), np.asarray(result.x)])
s_path = np.vstack([np.asarray(trace.s), np.asarray(result.s)])
z_path = np.vstack([np.asarray(trace.z), np.asarray(result.z)])
assert np.all(s_path > 0) and np.all(z_path > 0)

# Final KKT residual, recomputed here with plain NumPy.
xf, zf, sf = (np.asarray(v) for v in (result.x, result.z, result.s))
r_dual_f = P @ xf + q + G.T @ zf
r_ineq_f = G @ xf + sf - h
final_kkt = max(np.max(np.abs(r_dual_f)), np.max(np.abs(r_ineq_f)), float(sf @ zf))
assert final_kkt < 1e-7
assert np.max(np.abs(x_path[-1] - x_star)) < 1e-6 and np.min(s_path[-1]) < 1e-6
print(f"solved in {N} iterations; final x = {x_path[-1]}")

solved in 6 iterations; final x = [0.849999999 0.750000001]


In [3]:
mu = np.asarray(trace.mu)
mu_aff = np.asarray(trace.mu_aff)
sigma = np.asarray(trace.sigma)
pres = np.append(np.max(np.abs(np.asarray(trace.r_ineq)), axis=1), np.max(np.abs(r_ineq_f)))
dres = np.append(np.max(np.abs(np.asarray(trace.r_dual)), axis=1), np.max(np.abs(r_dual_f)))
alpha_p = np.asarray(trace.alpha_primal)
alpha_d = np.asarray(trace.alpha_dual)
lin_res = np.asarray(trace.linear_residual)

print("iter   primal      dual        mu         mu_aff      sigma      "
      "alpha_pri  alpha_dual  linear_res")
for k in range(N):
    print(f"{k:4d}  {pres[k]:.3e}  {dres[k]:.3e}  {mu[k]:.3e}  "
          f"{mu_aff[k]:.3e}  {sigma[k]:.3e}  "
          f"{alpha_p[k]:9.6f}  {alpha_d[k]:9.6f}  {lin_res[k]:.3e}")
print(f"{N} iterations, {result.factorizations} factorizations, "
      f"{result.newton_solves} Newton solves")
print(f"final KKT residual: {final_kkt:.2e}")
print(f"final complementarity s.z/5: {float(sf @ zf) / 5:.2e}")

iter   primal      dual        mu         mu_aff      sigma      alpha_pri  alpha_dual  linear_res
   0  0.000e+00  2.075e+00  1.000e+00  2.115e-01  9.467e-03   1.000000   0.982182  4.441e-16
   1  1.110e-16  4.264e-03  1.003e-01  1.555e-02  3.728e-03   0.846850   0.989182  1.110e-16
   2  1.110e-16  3.698e-02  6.855e-03  2.609e-04  5.516e-05   1.000000   0.980775  3.331e-16
   3  0.000e+00  8.427e-04  1.496e-04  8.865e-07  2.079e-07   0.990003   0.989977  1.332e-15
   4  1.110e-16  8.466e-06  1.503e-06  9.223e-11  2.310e-13   0.990000   0.990000  4.441e-16
   5  1.110e-16  8.466e-08  1.503e-08  9.224e-15  2.312e-19   0.990000   0.990000  5.551e-16
6 iterations, 6 factorizations, 12 Newton solves
final KKT residual: 8.47e-10
final complementarity s.z/5: 1.50e-10


The oracle for every gate below answers the same two questions with a fresh
dense solve of the full four-block system, never reusing a factorization:

$$\begin{bmatrix} P & A^\top & G^\top & 0\\ A & 0 & 0 & 0\\
G & 0 & 0 & I\\ 0 & 0 & S & Z \end{bmatrix}
\begin{bmatrix}\Delta x\\ \Delta y\\ \Delta z\\ \Delta s\end{bmatrix}
= \begin{bmatrix}-r_{\mathrm{dual}}\\ -r_{\mathrm{eq}}\\
-r_{\mathrm{ineq}}\\ c\end{bmatrix},
\qquad c = -s \circ z + \text{target} - \text{correction}.$$

The affine predictor sets target and correction to zero; the corrector sets
the target to $\sigma\mu\mathbf{1}$ and subtracts the predictor's cross term
$\Delta s_{\mathrm{aff}} \circ \Delta z_{\mathrm{aff}}$. Being deliberately
slow and independent is the point: agreeing final answers could hide
compensating errors, agreeing directions cannot.

In [4]:
TAU = 0.99  # the same fixed fraction-to-boundary damping the solver uses


def full_inf_norm(v):
    return jnp.max(jnp.abs(v), initial=0.0)  # initial=0 keeps an empty r_eq legal


def full_residuals(P, q, A, b, G, h, x, y, z, s):
    r_dual = P @ x + q + A.T @ y + G.T @ z
    r_eq = A @ x - b
    r_ineq = G @ x + s - h
    mu = s @ z / s.shape[0]
    return r_dual, r_eq, r_ineq, mu


def newton_full(P, A, G, z, s, r_dual, r_eq, r_ineq, c):
    n, m_eq, m_in = P.shape[0], A.shape[0], G.shape[0]
    O = jnp.zeros
    kkt = jnp.concatenate([
        jnp.concatenate([P, A.T, G.T, O((n, m_in))], axis=1),
        jnp.concatenate([A, O((m_eq, m_eq)), O((m_eq, m_in)), O((m_eq, m_in))], axis=1),
        jnp.concatenate([G, O((m_in, m_eq)), O((m_in, m_in)), jnp.eye(m_in)], axis=1),
        jnp.concatenate([O((m_in, n)), O((m_in, m_eq)), jnp.diag(s), jnp.diag(z)], axis=1),
    ], axis=0)
    rhs = jnp.concatenate([-r_dual, -r_eq, -r_ineq, c])
    d = jnp.linalg.solve(kkt, rhs)  # a fresh dense factorization for every direction
    return d[:n], d[n:n + m_eq], d[n + m_eq:n + m_eq + m_in], d[n + m_eq + m_in:]


def full_alpha(v, dv, tau):
    ratios = jnp.where(dv < 0, -v / dv, jnp.inf)
    return jnp.minimum(1.0, tau * jnp.min(ratios))


def reference_step(P, q, A, b, G, h, x, y, z, s, iteration):
    m_in = h.shape[0]
    r_dual, r_eq, r_ineq, mu = full_residuals(P, q, A, b, G, h, x, y, z, s)
    dx_aff, dy_aff, dz_aff, ds_aff = newton_full(
        P, A, G, z, s, r_dual, r_eq, r_ineq, -(s * z))
    alpha_aff_primal = full_alpha(s, ds_aff, 1.0)
    alpha_aff_dual = full_alpha(z, dz_aff, 1.0)
    mu_aff = (s + alpha_aff_primal * ds_aff) @ (z + alpha_aff_dual * dz_aff) / m_in
    sigma = jnp.clip((mu_aff / mu) ** 3, 0.0, 1.0)
    c_corr = -(s * z) + sigma * mu - ds_aff * dz_aff
    dx, dy, dz, ds = newton_full(P, A, G, z, s, r_dual, r_eq, r_ineq, c_corr)
    alpha_primal = full_alpha(s, ds, TAU)
    alpha_dual = full_alpha(z, dz, TAU)
    new = (x + alpha_primal * dx, y + alpha_dual * dy,
           z + alpha_dual * dz, s + alpha_primal * ds)
    snap = dict(iteration=iteration, x=x, y=y, z=z, s=s, mu=mu,
                dx_aff=dx_aff, dy_aff=dy_aff, dz_aff=dz_aff, ds_aff=ds_aff,
                alpha_aff_primal=alpha_aff_primal, alpha_aff_dual=alpha_aff_dual,
                mu_aff=mu_aff, sigma=sigma, dx=dx, dy=dy, dz=dz, ds=ds,
                alpha_primal=alpha_primal, alpha_dual=alpha_dual)
    return new, snap


def reference_solve(P, q, A, b, G, h, eps_abs=1e-8, eps_rel=1e-8, max_iter=50):
    P, q, A, b, G, h = map(jnp.asarray, (P, q, A, b, G, h))
    x, y = jnp.zeros(q.shape[0]), jnp.zeros(b.shape[0])
    z, s = jnp.ones(h.shape[0]), jnp.ones(h.shape[0])
    snaps, status = [], "max_iter"
    for iteration in range(max_iter):
        (x, y, z, s), snap = reference_step(P, q, A, b, G, h, x, y, z, s, iteration)
        snaps.append(snap)
        # the same absolute-plus-relative stop test the solver applies,
        # judged only on the newly accepted iterate
        r_dual, r_eq, r_ineq, _ = full_residuals(P, q, A, b, G, h, x, y, z, s)
        eps_p = eps_abs + eps_rel * jnp.max(jnp.array(
            [full_inf_norm(A @ x), full_inf_norm(b), full_inf_norm(G @ x),
             full_inf_norm(s), full_inf_norm(h)]))
        eps_d = eps_abs + eps_rel * jnp.max(jnp.array(
            [full_inf_norm(P @ x), full_inf_norm(q), full_inf_norm(A.T @ y),
             full_inf_norm(G.T @ z)]))
        eps_g = eps_abs + eps_rel * jnp.abs(0.5 * x @ P @ x + q @ x)
        pres = jnp.maximum(full_inf_norm(r_eq), full_inf_norm(r_ineq))
        if bool((pres <= eps_p) & (full_inf_norm(r_dual) <= eps_d) & (s @ z <= eps_g)):
            status = "solved"
            break
    return (x, y, z, s), status, snaps

In [5]:
F = fractions.Fraction

# tiny: hand-checkable QP with one equality; optimum x = (0, 1), y = -2,
# z = (0, 1), s = (2, 0). Integer data keeps the first iteration rational.
tiny = (np.array([[4.0, 1.0], [1.0, 3.0]]), np.array([1.0, -2.0]),
        np.array([[1.0, 1.0]]), np.array([1.0]),
        np.array([[1.0, 0.0], [0.0, 1.0]]), np.array([2.0, 1.0]))


def exact_solve(M, rhs):
    """Gauss-Jordan over exact rationals: any nonzero pivot is exact."""
    M, rhs = [row[:] for row in M], rhs[:]
    for col in range(len(M)):
        piv = next(r for r in range(col, len(M)) if M[r][col] != 0)
        M[col], M[piv], rhs[col], rhs[piv] = M[piv], M[col], rhs[piv], rhs[col]
        for r in range(len(M)):
            if r != col and M[r][col] != 0:
                f = M[r][col] / M[col][col]
                M[r] = [a - f * bb for a, bb in zip(M[r], M[col])]
                rhs[r] -= f * rhs[col]
    return [rhs[r] / M[r][r] for r in range(len(M))]


def exact_alpha(v, dv, tau):
    ratios = [-vi / dvi for vi, dvi in zip(v, dv) if dvi < 0]
    return min(F(1), tau * min(ratios)) if ratios else F(1)


# First iteration of tiny in exact arithmetic, from the start x=0, y=0, s=z=1:
# r_dual = q + G.T z = [2, -1], r_eq = [-1], r_ineq = [-1, 0], mu = 1.
x0, y0 = [F(0), F(0)], [F(0)]
z0, s0 = [F(1), F(1)], [F(1), F(1)]
r_dual0, r_eq0, r_ineq0, mu0 = [F(2), F(-1)], [F(-1)], [F(-1), F(0)], F(1)
kkt7 = [  # the four-block matrix of the reference, with S = Z = I
    [F(4), F(1), F(1), F(1), F(0), F(0), F(0)],
    [F(1), F(3), F(1), F(0), F(1), F(0), F(0)],
    [F(1), F(1), F(0), F(0), F(0), F(0), F(0)],
    [F(1), F(0), F(0), F(0), F(0), F(1), F(0)],
    [F(0), F(1), F(0), F(0), F(0), F(0), F(1)],
    [F(0), F(0), F(0), F(1), F(0), F(1), F(0)],
    [F(0), F(0), F(0), F(0), F(1), F(0), F(1)],
]


def exact_direction(c):
    d = exact_solve(kkt7, [-r for r in r_dual0] + [F(1)] + [F(1), F(0)] + c)
    return d[0:2], d[2:3], d[3:5], d[5:7]

dx_aff, dy_aff, dz_aff, ds_aff = exact_direction([F(-1), F(-1)])
a_p = exact_alpha(s0, ds_aff, F(1))
a_d = exact_alpha(z0, dz_aff, F(1))
mu_aff_e = sum((si + a_p * di) * (zi + a_d * dj)
               for si, di, zi, dj in zip(s0, ds_aff, z0, dz_aff)) / 2
sigma_e = min(max((mu_aff_e / mu0) ** 3, F(0)), F(1))
c_corr = [-si * zi + sigma_e * mu0 - di * dj
          for si, zi, di, dj in zip(s0, z0, ds_aff, dz_aff)]
dx_c, dy_c, dz_c, ds_c = exact_direction(c_corr)
a_pc = exact_alpha(s0, ds_c, F(99, 100))
a_dc = exact_alpha(z0, dz_c, F(99, 100))

derived = {
    "dx_aff": dx_aff, "dy_aff": dy_aff, "dz_aff": dz_aff, "ds_aff": ds_aff,
    "alpha_aff_primal": a_p, "alpha_aff_dual": a_d,
    "mu_aff": mu_aff_e, "sigma": sigma_e,
    "dx": dx_c, "dy": dy_c, "dz": dz_c, "ds": ds_c,
    "alpha_primal": a_pc, "alpha_dual": a_dc,
}
expected = {
    "dx_aff": [F(1, 7), F(6, 7)], "dy_aff": [F(-11, 7)],
    "dz_aff": [F(-13, 7), F(-1, 7)], "ds_aff": [F(6, 7), F(-6, 7)],
    "alpha_aff_primal": F(1), "alpha_aff_dual": F(7, 13),
    "mu_aff": F(6, 91), "sigma": F(216, 753571),
    "dx": [F(-5, 49), F(54, 49)], "dy": [F(-1645769, 753571)],
    "dz": [F(-384259, 753571), F(-15163, 753571)], "ds": [F(54, 49), F(-54, 49)],
    "alpha_primal": F(539, 600), "alpha_dual": F(1),
}
assert derived == expected

# Both float implementations must reproduce the exact rationals.
start = [jnp.asarray(np.asarray(v, dtype=np.float64)) for v in (x0, y0, z0, s0)]
_, snap0 = reference_step(*[jnp.asarray(v) for v in tiny], *start, 0)
result_tiny, trace_tiny = barrierqp.Solver(*tiny).trace()
for key, value in expected.items():
    target = np.asarray(value, dtype=np.float64)
    assert_allclose(np.asarray(snap0[key]), target, rtol=1e-13, atol=1e-15,
                    err_msg=key)
    assert_allclose(np.asarray(getattr(trace_tiny, key)[0]), target,
                    rtol=1e-13, atol=1e-15, err_msg=key)
print("audit okay: first tiny iteration matches exact fraction arithmetic")

audit okay: first tiny iteration matches exact fraction arithmetic


In [6]:
# big: seeded strictly convex QP around an attainable interior point, so the
# equalities are consistent and Slater holds by construction.
rng = np.random.default_rng(0)
n, m_eq, m_in = 6, 2, 8
M = rng.normal(size=(n, n))
A_big = rng.normal(size=(m_eq, n))
x_feas = rng.normal(size=n)
G_big = rng.normal(size=(m_in, n))
big = (M.T @ M + np.eye(n), rng.normal(size=n),
       A_big, A_big @ x_feas,
       G_big, G_big @ x_feas + rng.uniform(0.1, 1.0, size=m_in))

# noeq: no equality rows at all (m_eq = 0), like the pentagon.
noeq = (np.array([[2.0, 0.5], [0.5, 1.0]]), np.array([-2.0, -2.0]),
        np.zeros((0, 2)), np.zeros(0),
        np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]), np.ones(3))

fixtures = {"pentagon": (P, q, A, b, G, h), "tiny": tiny, "big": big, "noeq": noeq}

DIRECTION_FIELDS = ["dx_aff", "dy_aff", "dz_aff", "ds_aff", "dx", "dy", "dz", "ds"]
SCALAR_FIELDS = ["alpha_aff_primal", "alpha_aff_dual", "mu_aff", "sigma",
                 "alpha_primal", "alpha_dual", "mu"]
STATE_FIELDS = ["x", "y", "z", "s"]

results, traces, worst, counts = {}, {}, {}, {}
for name, prob in fixtures.items():
    ref_final, ref_status, ref_snaps = reference_solve(*prob)
    res, tr = barrierqp.Solver(*prob).trace()
    assert res.status == ref_status == "solved"
    assert res.iterations == len(ref_snaps)
    diff = 0.0
    for k, snap in enumerate(ref_snaps):
        for field in DIRECTION_FIELDS + SCALAR_FIELDS + STATE_FIELDS:
            ref_v = np.asarray(snap[field])
            red_v = np.asarray(getattr(tr, field)[k])
            if ref_v.size:
                assert_allclose(red_v, ref_v, rtol=1e-9, atol=1e-11,
                                err_msg=f"{name} {field}")
                if field in DIRECTION_FIELDS:
                    diff = max(diff, float(np.max(np.abs(red_v - ref_v))))
    results[name], traces[name] = res, tr
    worst[name], counts[name] = diff, res.iterations

assert counts == {"pentagon": 6, "tiny": 5, "big": 7, "noeq": 6}
print(f"directions okay: pentagon max full/reduced difference {worst['pentagon']:.2e}, "
      f"all fixtures {max(worst.values()):.2e}")
print(f"trajectories okay: iterations {counts}")

directions okay: pentagon max full/reduced difference 2.09e-15, all fixtures 1.93e-12
trajectories okay: iterations {'pentagon': 6, 'tiny': 5, 'big': 7, 'noeq': 6}


In [7]:
for name, prob in fixtures.items():
    res, tr = results[name], traces[name]
    # strict interiority of every recorded iterate and the accepted end
    assert np.all(np.asarray(tr.s) > 0) and np.all(np.asarray(tr.z) > 0)
    assert np.all(np.asarray(res.s) > 0) and np.all(np.asarray(res.z) > 0)
    # the reused LU factor kept answering sharply on every iteration
    assert float(np.max(np.asarray(tr.linear_residual))) < 1e-10

    # the compiled while_loop must land where the traced loop landed
    fresh = barrierqp.Solver(*prob)
    compiled = fresh.solve()
    assert compiled.status == res.status and compiled.iterations == res.iterations
    for field in STATE_FIELDS:
        assert_allclose(np.asarray(getattr(compiled, field)),
                        np.asarray(getattr(res, field)), rtol=1e-9, atol=1e-11)

    # the jitted step is the eager step, staged once
    st0 = barrierqp.init_state(fresh.problem)
    st_e, tr_e = barrierqp.step(fresh.problem, st0, 1e-8, 1e-8)
    st_j, tr_j = barrierqp._step(fresh.problem, st0, 1e-8, 1e-8)
    for field in DIRECTION_FIELDS + SCALAR_FIELDS:
        v = np.asarray(getattr(tr_e, field))
        if v.size:
            assert_allclose(np.asarray(getattr(tr_j, field)), v,
                            rtol=1e-12, atol=1e-14, err_msg=f"{name} {field}")
    # the loop carry keeps fixed shapes and dtypes, iteration after iteration
    assert jax.tree.map(lambda a: (a.shape, a.dtype), st0) == \
           jax.tree.map(lambda a: (a.shape, a.dtype), st_j)

    # count bookkeeping: one factorization, two Newton solves per iteration
    assert res.factorizations == res.iterations
    assert res.newton_solves == 2 * res.iterations
print("jax checks okay: eager/jit, compiled/traced, positivity, "
      "linear residuals, fixed shapes, N/2N counts")

jax checks okay: eager/jit, compiled/traced, positivity, linear residuals, fixed shapes, N/2N counts


In [8]:
# OSQP sees the same QP as  l <= [A; G] x <= u  with l = [b, -inf], u = [b, h].
for name, prob in fixtures.items():
    Pf, qf, Af, bf, Gf, hf = (np.asarray(v, dtype=np.float64) for v in prob)
    stacked = sparse.csc_matrix(np.vstack([Af, Gf]))
    lower = np.hstack([bf, np.full(hf.shape, -np.inf)])
    upper = np.hstack([bf, hf])
    oracle = osqp.OSQP()
    oracle.setup(sparse.csc_matrix(Pf), qf, stacked, lower, upper,
                 eps_abs=1e-9, eps_rel=1e-9, verbose=False, max_iter=200000)
    osqp_res = oracle.solve()
    assert osqp_res.info.status == "solved"

    res = results[name]
    x = np.asarray(res.x)
    objective = 0.5 * x @ Pf @ x + qf @ x
    assert_allclose(objective, osqp_res.info.obj_val, rtol=1e-7, atol=1e-7)
    assert_allclose(x, osqp_res.x, rtol=1e-5, atol=1e-5)

    # our final point satisfies the original KKT system on its own terms
    yv, zv, sv = (np.asarray(v) for v in (res.y, res.z, res.s))
    kkt = max(np.max(np.abs(Pf @ x + qf + Af.T @ yv + Gf.T @ zv)),
              np.max(np.abs(Gf @ x + sv - hf)),
              np.max(np.abs(Af @ x - bf), initial=0.0), float(sv @ zv))
    assert kkt < 1e-7
print("OSQP okay: objectives, solutions, and KKT residuals agree on all fixtures")

OSQP okay: objectives, solutions, and KKT residuals agree on all fixtures


Deliberately not implemented: Phase I initialization, infeasibility or
unboundedness certificates, homogeneous embedding, scaling, regularization,
iterative refinement, sparse or OCP structure, cones, warm starts, an
autodiff layer, and any production promise. The pentagon above is the whole
story this project tells: one factorization per iteration, and two Newton
questions answered from it.